In [1]:
class Process:
    def _init_(self, pid, arrival_time, burst_time, priority=0):
        self.pid = pid
        self.arrival_time = arrival_time
        self.burst_time = burst_time
        self.priority = priority
        self.remaining_time = burst_time
        self.start_time = None
        self.completion_time = None


In [2]:
# FCFS Scheduling
def fcfs(processes):
    processes.sort(key=lambda p: p.arrival_time)
    current_time = 0
    schedule = []
    for process in processes:
        process.start_time = max(current_time, process.arrival_time)
        current_time = process.start_time + process.burst_time
        process.completion_time = current_time
        schedule.append(process)
    return schedule


In [3]:
def sjf(processes):
    processes.sort(key=lambda x: (x.arrival_time, x.burst_time))
    completed, current_time, schedule = [], 0, []
    while processes:
        available = [p for p in processes if p.arrival_time <= current_time]
        if not available:
            current_time = processes[0].arrival_time
            continue
        process = min(available, key=lambda p: p.burst_time)
        processes.remove(process)
        process.start_time = current_time
        current_time += process.burst_time
        process.completion_time = current_time
        schedule.append(process)
    return schedule


In [4]:
def priority_scheduling(processes):
    processes.sort(key=lambda x: (x.arrival_time, x.priority))
    current_time, schedule = 0, []
    while processes:
        available = [p for p in processes if p.arrival_time <= current_time]
        if not available:
            current_time = processes[0].arrival_time
            continue
        process = min(available, key=lambda p: p.priority)
        processes.remove(process)
        process.start_time = current_time
        current_time += process.burst_time
        process.completion_time = current_time
        schedule.append(process)
    return schedule


In [ ]:
def round_robin(processes, quantum):
    processes.sort(key=lambda p: p.arrival_time)
    queue, schedule, current_time = processes[:], [], processes[0].arrival_time
    queue = [p for p in queue if p.arrival_time <= current_time]
    queue.sort(key=lambda p: p.arrival_time)
    num_of_p_to_q = len(queue)
    while queue:
        process = queue.pop(0)
        schedule.append(process.pid)

        if process.arrival_time > current_time:
            current_time = process.arrival_time
        execution_time = min(quantum, process.remaining_time)
        process.remaining_time -= execution_time
        current_time += execution_time
        if process.remaining_time == 0:
            process.completion_time = current_time
        else:
            queue.append(process)
        l = [p for p in processes if p.arrival_time <= current_time and p not in queue and p.remaining_time > 0]
        l.sort(key=lambda p: p.arrival_time)
        queue = l + queue

        print("queue", [p.pid for p in queue])
        print("current_time", current_time)
        print("scheduel", schedule)
        
    return schedule



In [33]:
def priority_rr(processes,quantum):
    processes.sort(key=lambda p: p.arrival_time)
    queue, schedule, current_time = processes[:], [], processes[0].arrival_time
    queue = [p for p in queue if p.arrival_time <= current_time]
    queue = [p for p in queue if p.priority == min([p.priority for p in queue])]
    while queue:
        process = queue.pop(0)
        schedule.append(process.pid)

        if process.arrival_time > current_time:
            current_time = process.arrival_time
        execution_time = min(quantum, process.remaining_time)
        process.remaining_time -= execution_time
        current_time += execution_time
        if process.remaining_time == 0:
            process.completion_time = current_time
        else:
            queue.append(process)
        l = [p for p in processes if p.arrival_time <= current_time and p not in queue and p.remaining_time > 0]
        queue.extend(l)
        queue = [p for p in queue if p.priority == min([p.priority for p in queue])]
        print("queue", [p.pid for p in queue])
        print("current_time", current_time)
        print("schedule", schedule)
        
    return schedule

p1 = Process()
p1._init_(1, 5, 8,1)
p2 = Process()
p2._init_(2, 0, 4,3)
p3 = Process()
p3._init_(3, 0, 2,1)
p4 = Process()
p4._init_(4, 1, 6,4)

print(priority_rr([p1, p2, p3, p4], 2))


queue [2]
current_time 2
schedule [3]
queue [2]
current_time 4
schedule [3, 2]
queue [1]
current_time 6
schedule [3, 2, 2]
queue [1]
current_time 8
schedule [3, 2, 2, 1]
queue [1]
current_time 10
schedule [3, 2, 2, 1, 1]
queue [1]
current_time 12
schedule [3, 2, 2, 1, 1, 1]
queue [4]
current_time 14
schedule [3, 2, 2, 1, 1, 1, 1]
queue [4]
current_time 16
schedule [3, 2, 2, 1, 1, 1, 1, 4]
queue [4]
current_time 18
schedule [3, 2, 2, 1, 1, 1, 1, 4, 4]
queue []
current_time 20
schedule [3, 2, 2, 1, 1, 1, 1, 4, 4, 4]
[3, 2, 2, 1, 1, 1, 1, 4, 4, 4]


In [ ]:
import json
import sys

def parse_path(path):
    components = []
    if not path.startswith('$'):
        return None
    remaining = path[1:]  # strip the $
    i = 0
    n = len(remaining)
    while i < n:
        if remaining[i] == '.':
            i += 1
            start = i
            while i < n and (remaining[i].isalnum() or remaining[i] == '_'):
                i += 1
            component = remaining[start:i]
            components.append(component)
        elif remaining[i] == '[':
            i += 1
            start = i
            while i < n and remaining[i].isdigit():
                i += 1
            if i >= n or remaining[i] != ']':
                return None  # invalid path
            component = remaining[start:i]
            components.append(component)
            i += 1
        else:
            return None  # invalid path
    return components

def query_json(data, path_components):
    current = data
    for component in path_components:
        if isinstance(current, dict):
            if component in current:
                current = current[component]
            else:
                return None
        elif isinstance(current, list):
            if component.isdigit():
                index = int(component)
                if 0 <= index < len(current):
                    current = current[index]
                else:
                    return None
            else:
                return None
        else:
            return None
    return current

def format_output(value):
    if value is None:
        return "None"
    elif isinstance(value, (dict, list)):
        # For dictionaries, output in the original key order (Python 3.7+ preserves insertion order)
        if isinstance(value, dict):
            items = []
            for k, v in value.items():
                if isinstance(v, str):
                    v_str = f'"{v}"'
                else:
                    v_str = str(v)
                items.append(f'"{k}": {v_str}')
            return '{' + ','.join(items) + '}'
        elif isinstance(value, list):
            items = []
            for item in value:
                if isinstance(item, str):
                    items.append(f'"{item}"')
                elif isinstance(item, (dict, list)):
                    items.append(format_output(item))
                else:
                    items.append(str(item))
            return '[' + ','.join(items) + ']'
    elif isinstance(value, str):
        return f'"{value}"'
    else:
        return str(value)

def main():
    json_str = sys.stdin.readline().strip()
    data = json.loads(json_str)
    q = int(sys.stdin.readline())
    for _ in range(q):
        path = sys.stdin.readline().strip()
        components = parse_path(path)
        if components is None:
            print("None")
            continue
        result = query_json(data, components)
        if result is None:
            print("None")
        else:
            print(format_output(result))

if __name__ == '__main__':
    main()